<div style="display: flex; justify-content: flex-start; align-items: center;">
    <a href="https://colab.research.google.com/github/msfasha/307307-BI-Methods-Generative-AI/blob/main/20261/Module%206%20-%20Introduction%20to%20CrewAI/CrewAI_1_Basics_lab.ipynb" target="_blank">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 25px;">
    </a>
</div>

<div style="background-color:#F5EDF4; border-top:3px solid #A83296; padding:16px; border-radius:6px">
<h1 style="color:#493347; margin:0">Lab 1: CrewAI Basics</h1>
<p style="color:#56616D; margin:6px 0 0 0">307307 BI Methods for Generative AI - University of Petra</p>
</div>

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Our plan: three labs, one question</h2>
</div>

Over three labs we ask **one business question**, and change only **where the facts come from**. Watch how the answer changes.

> **Petra Bean** is a small (fictional) neighbourhood cafe in Amman, Jordan. The owner is thinking about selling **bottled cold brew coffee (330 ml)** and asks: *should we launch it?*

| Lab | Where the facts come from | What you learn |
|---|---|---|
| **1. Basics (this lab)** | **The language model's memory.** No data, no search. | The building blocks: LLM, Agent, Task, Crew, Process, Tool |
| **2. Cafe lab** | **Our own files:** sales, market notes, the owner's handbook | Tools on real data, knowledge, guardrails, flows |
| **3. Web search lab** | **The live web** (Tavily search) | Search tools, citing sources, why web text is untrusted |

In this lab the team can only say what the model already "knows". The answer will sound confident but cannot be checked. That is the starting point; Labs 2 and 3 make it more real.

**Keep your Lab 1 answer.** You will compare it with the answers from Labs 2 and 3.

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Objectives</h2>
</div>

Build a small team of two AI agents and learn the six CrewAI building blocks:

| Building block | Meaning | In this lab |
|---|---|---|
| **LLM** | The language model behind the agents | Gemini |
| **Agent** | An AI worker with a role, goal and backstory | Market researcher, strategist |
| **Task** | One unit of work, given to one agent | Market overview, recommendation |
| **Crew** | The agents and tasks, run together | `cafe_crew` |
| **Process** | The order in which tasks run | `Process.sequential` |
| **Tool** | A Python function an agent can call | Cold brew brief lookup |

> The agents do not look anything up. Their answers come from what the model already knows, so treat them as illustrative, not as verified research.

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Setup</h2>
</div>

Install CrewAI with Gemini support, then give it your Gemini API key. In Google Colab, store the key as `GOOGLE_API_KEY` in the Secrets panel (the key icon on the left). On your own computer, keep it in a `.env` file or let the cell ask you to paste it. Get a key from [Google AI Studio](https://aistudio.google.com/app/apikey), and never put a key inside a notebook you share.

> **Free keys are limited.** At the time of writing, a free Gemini key allows only a handful of requests per minute and about twenty per day for some models. A crew makes several requests per run, so you may hit `429` errors. If that happens, wait, or use a key with billing enabled (these labs cost very little to run).

In [ ]:
# Install CrewAI with Gemini support.
%pip install -q "crewai[google-genai]"

In [ ]:
import os, getpass
from dotenv import load_dotenv, find_dotenv

# If you keep your keys in a .env file (local use), load them. On Colab there is no .env, so this does nothing.
load_dotenv(find_dotenv(usecwd=True))

# Read the API key from Colab Secrets if available, otherwise from the environment, otherwise ask.
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    if not os.environ.get("GOOGLE_API_KEY"):
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")

# Turn off anonymous usage reporting.
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["CREWAI_TRACING_ENABLED"] = "false"

print("API key loaded:", bool(os.environ.get("GOOGLE_API_KEY")))

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Building block 1: LLM</h2>
</div>

CrewAI talks to many providers. The prefix `gemini/` selects Google Gemini. If you get a "model not found" error, change only this line to a model your key can use.

In [ ]:
from crewai import LLM

llm = LLM(model="gemini/gemini-3.8-flash")   # change this line if your key cannot see this model

# Quick test: one direct question to the model, no agents yet.
print(llm.call("Say hello to a coffee shop owner in one short sentence."))

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Building block 2: Agent</h2>
</div>

An **Agent** is an AI worker. Three sentences describe it, and CrewAI turns them into the instructions the LLM receives:

- `role`: who the agent is.
- `goal`: what it should achieve.
- `backstory`: background that guides its behaviour.

Write them like a job description. `verbose=True` prints the agent's thinking so you can follow it.

In [ ]:
from crewai import Agent

market_researcher = Agent(
    role="Market Research Analyst",
    goal="Prepare a clear, balanced overview of a product opportunity for a small cafe.",
    backstory=(
        "You are a careful market research analyst who advises independent cafes. "
        "You separate assumptions from facts and write for a business audience."
    ),
    llm=llm,
    verbose=True,
)

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Building block 3: Task</h2>
</div>

A **Task** describes the work: a `description` (what to do), an `expected_output` (what a good answer looks like) and the `agent` who does it.

In [ ]:
from crewai import Task

research_task = Task(
    description=(
        "Petra Bean is a small neighbourhood cafe in Amman, Jordan. The owner is considering "
        "selling bottled cold brew coffee (330 ml). Analyse the opportunity: target customers, "
        "demand, competitors, typical prices, opportunities and risks. Label assumptions clearly. "
        "Do not invent statistics or claim to have done live research."
    ),
    expected_output=(
        "A concise market overview with sections: target customers, demand, "
        "competitors, prices, opportunity, risks, assumptions."
    ),
    agent=market_researcher,
)

### A second agent and task

The strategist receives the researcher's result through `context=[research_task]`. This is how one task passes its work to the next.

In [ ]:
business_strategist = Agent(
    role="Business Strategy Consultant",
    goal="Turn the market overview into a practical launch recommendation for the cafe owner.",
    backstory=(
        "You are a pragmatic consultant who looks at customer value, "
        "differentiation, feasibility and business risk."
    ),
    llm=llm,
    verbose=True,
)

strategy_task = Task(
    description=(
        "Using the market overview, recommend whether Petra Bean should launch bottled cold brew. "
        "Propose a small pilot: what to sell, at what kind of price, for how long, "
        "and what result would tell the owner to continue or stop. Include the key risks."
    ),
    expected_output=(
        "A short recommendation: go / maybe / no-go, the pilot plan, "
        "the signal to continue or stop, and the key risks."
    ),
    agent=business_strategist,
    context=[research_task],     # receives the result of the first task
)

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Building blocks 4 and 5: Crew and Process</h2>
</div>

A **Crew** puts agents and tasks together. Its **Process** says how the tasks run:

- `Process.sequential`: one task after another, in the order given (an assembly line).
- `Process.hierarchical`: a manager agent hands out the work (we do not use it in these labs).

We use sequential: research first, then strategy.

In [ ]:
from crewai import Crew, Process

cafe_crew = Crew(
    agents=[market_researcher, business_strategist],
    tasks=[research_task, strategy_task],
    process=Process.sequential,
    verbose=True,
)

### Run the crew

`kickoff_async` starts the work. It sends requests to Gemini, so it uses some of your API quota and takes a minute. In a notebook we write `await crew.kickoff_async()`; in an ordinary Python script you would write `crew.kickoff()`.

In [ ]:
result = await cafe_crew.kickoff_async()

print("\n===== FINAL OUTPUT =====\n")
print(result)

### What happened

1. The Market Research Analyst completed `research_task`.
2. The Business Strategy Consultant received that result through `context` and completed `strategy_task`.
3. The crew returned the last task's result.

```text
Crew = Agents + Tasks + Process

Researcher --(Task 1: market overview)--> Strategist --(Task 2: recommendation)--> Final result
```

**Read the answer critically.** Where do its numbers (prices, percentages, market sizes) come from? The agents have no data, so anything specific is a guess. Save this answer: Labs 2 and 3 answer the same question with real facts.

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Building block 6: Tool</h2>
</div>

A language model only produces text. A **Tool** is a Python function the agent can call to do more: calculate, read a file, query a database or search the web. The agent reads the tool's name and the description in triple quotes, decides when to use it, and gets the result back.

Here the tool is a **stand-in**: it returns a fixed sample brief. It is not a live data source. In Lab 2 we replace it with tools that read the cafe's real files, and in Lab 3 with a live web search.

In [ ]:
from crewai.tools import tool

@tool("Cold brew brief lookup")
def get_cold_brew_brief(topic: str) -> str:
    """Return a sample brief about the bottled cold brew idea for a small cafe."""
    return (
        "Sample brief: customers may value convenience, taste and a caffeine kick. "
        "Competitors include supermarket cold brew (cheaper) and specialty cafes (pricier, higher quality). "
        "Bottles need cold storage and a short shelf life. "
        "This is illustrative input, not verified market research."
    )

# A tool is an ordinary function: test it directly, without any AI.
print(get_cold_brew_brief.run("bottled cold brew"))

Give the tool to an agent with `tools=[...]`. The agent decides when to use it, and the log shows each call.

In [ ]:
tool_agent = Agent(
    role="Product Analyst",
    goal="Use the product brief to summarise what matters for the cafe.",
    backstory="You are a concise analyst who uses the tools you are given.",
    llm=llm,
    tools=[get_cold_brew_brief],     # the tool, given to the agent
    verbose=True,
)

tool_task = Task(
    description=(
        "Use the Cold brew brief lookup tool for 'bottled cold brew'. "
        "Summarise what it returns and state its limitations."
    ),
    expected_output="A short summary based on the tool result, including limitations.",
    agent=tool_agent,
)

tool_crew = Crew(agents=[tool_agent], tasks=[tool_task], process=Process.sequential, verbose=True)

tool_result = await tool_crew.kickoff_async()
print(tool_result)

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Knowledge check</h2>
</div>

1. What is the difference between an Agent and a Task?
2. What do `role`, `goal` and `backstory` describe?
3. What does a Crew contain?
4. What is the difference between a sequential and a hierarchical process?
5. Why might an agent need a Tool?
6. Why can we not trust the specific numbers in this lab's answer?

<details>
<summary>Suggested answers</summary>

1. An Agent does work; a Task defines the work to be done.
2. The agent's job, its objective, and background that guides its behaviour.
3. Agents and tasks, run under a process.
4. Sequential runs tasks in a fixed order; hierarchical uses a manager agent to hand out the work.
5. To act or get information beyond generating text (calculate, read data, search).
6. Because the agents have no data source. Their answers come from the model's general knowledge, so any number may be a guess.
</details>

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Summary</h2>
</div>

| Building block | Meaning | In this lab |
|---|---|---|
| LLM | Language model behind the agents | Gemini |
| Agent | AI worker with role, goal, backstory | Researcher, strategist |
| Task | One unit of work | Market overview, recommendation |
| Crew | Agents and tasks run together | `cafe_crew` |
| Process | Order of the work | `Process.sequential` |
| Tool | Function an agent can call | Cold brew brief lookup (a stand-in) |

**The point of this lab:** a team of agents can produce a fluent recommendation with no data at all. That is exactly why it cannot be trusted yet.

**Next:** Lab 2 (`CrewAI_2_Cafe_lab.ipynb`) answers the same question using the cafe's own sales data, market notes and rules.